# 02 · Ingest raw World Bank data into BRONZE
**Goal:** land raw data *exactly as received* in the governed Volume `/Volumes/procintel/bronze/raw_files/`.

Sources (all public World Bank APIs):
| Layer | Source | Content |
|---|---|---|
| Structured | Projects API, Contract awards, CPIA/WGI indicators | what happened |
| Semi-structured | Procurement notices | how it was done |
| Unstructured | Documents & Reports API (ICR, PAD, ISR text) | why it succeeded or failed |

Step 1: test whether this compute can reach the internet (egress test).

In [0]:
# =====================================================================
# CELL 1: SETUP -> HTTP session with retries + a JSON helper
# =====================================================================
# (API exploration that led to these choices lives in 00_api_exploration:
#  verified filters, the renamed WGI codes, the contracts endpoint.)
import json
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Session = re-used connection; Retry = wait 1.5s, 3s, 6s, 12s on 429/5xx (exponential backoff)
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=4, backoff_factor=1.5, status_forcelist=[429, 500, 502, 503, 504])))

def get_json(url, params=None):
    """Call an API and return its JSON answer as Python objects. Raises on HTTP errors."""
    response = session.get(url, params=params, timeout=60)
    response.raise_for_status()
    return response.json()

In [0]:
# =====================================================================
# CELL 8: THE REAL DOWNLOAD -> raw World Bank data into the BRONZE Volume
# =====================================================================
# Design:
#  * ONE reusable function pages through any search.worldbank.org API
#    (projects, contracts, notices, documents all share the same paging).
#  * Data is saved RAW (exactly as received) as JSONL = one JSON record per line.
#  * Each run goes into its own dated folder: run_date=YYYY-MM-DD
#    -> keeps HISTORY (we can compare runs) and re-running on the same
#       day just overwrites that day's files (idempotent).
#  * After every download we VERIFY: rows received == total promised.

import os, datetime

RUN_DATE = datetime.date.today().isoformat()            # e.g. "2026-11-05"
LANDING  = "/Volumes/procintel/bronze/raw_files"        # our governed Volume

def extract_records(answer, record_key):
    """The APIs return records either as a LIST or as a DICT keyed by id.
    This function always returns a plain list of records."""
    block = answer.get(record_key, [])
    if isinstance(block, dict):        # projects & documents: {"P123": {...}, ...}
        return [v for v in block.values() if isinstance(v, dict) and "id" in v]
    return block                       # contracts & notices: already a list

def download_search_api(name, url, record_key, filters, page_size=500):
    """Page through a search.worldbank.org API until all records are fetched.
    Paging: rows = page size, os = offset (how many records to skip)."""
    records, offset = [], 0
    while True:                                        # loop until we 'break'
        answer = get_json(url, {"format": "json", "rows": page_size, "os": offset, **filters})
        total  = int(answer["total"])                  # some APIs send it as text
        page   = extract_records(answer, record_key)
        if not page:                                   # empty page -> nothing left
            break
        records.extend(page)                           # add this page to our list
        # IMPORTANT: move forward by what we RECEIVED, not by what we ASKED for.
        # If the server silently caps pages at 100, offset += 500 would SKIP data.
        offset += len(page)
        print(f"  {name}: {len(records):>6} / {total}")
        if offset >= total:
            break
    return records, total

def save_jsonl(records, source):
    """Write records as JSON Lines into a dated folder in the Volume."""
    folder = f"{LANDING}/{source}/run_date={RUN_DATE}"
    os.makedirs(folder, exist_ok=True)                 # create folders if missing
    path = f"{folder}/{source}.jsonl"
    with open(path, "w") as f:                         # "w" = write (overwrite)
        for record in records:
            f.write(json.dumps(record) + "\n")         # one record per line
    return path

# ---- The 4 search-platform sources, with the filters we VERIFIED ------
SOURCES = [
    # (name,       url,                                          record key,   Kenya filter)
    ("projects",  "https://search.worldbank.org/api/v2/projects", "projects",    {"countrycode_exact": "KE"}),
    ("contracts", "https://search.worldbank.org/api/contractdata","contract",    {"countryshortname_exact": "Kenya"}),
    ("notices",   "https://search.worldbank.org/api/v2/procnotices","procnotices",{"project_ctry_name_exact": "Kenya"}),
    ("documents", "https://search.worldbank.org/api/v2/wds",      "documents",   {"count_exact": "Kenya"}),
]

summary = []                                           # one row per source, for the final report
for name, url, key, filters in SOURCES:
    print(f"⬇️  {name} ...")
    records, total = download_search_api(name, url, key, filters)
    path = save_jsonl(records, name)
    status = "✅" if len(records) == total else "⚠️ MISMATCH"
    summary.append((name, len(records), total, status, path))

# ---- Governance indicators (different API, tiny: 4 indicators x ~27 years)
INDICATORS = {
    "IQ.CPA.PUBS.XQ": "CPIA public sector management & institutions (1-6)",
    "GOV_WGI_GE.EST": "WGI Government Effectiveness (-2.5..+2.5)",
    "GOV_WGI_RQ.EST": "WGI Regulatory Quality (-2.5..+2.5)",
    "GOV_WGI_CC.EST": "WGI Control of Corruption (-2.5..+2.5)",
}
indicator_rows = []
for code in INDICATORS:
    meta, rows = get_json(f"https://api.worldbank.org/v2/country/KEN/indicator/{code}",
                          {"format": "json", "per_page": 1000})
    indicator_rows.extend(rows)
path = save_jsonl(indicator_rows, "indicators")
summary.append(("indicators", len(indicator_rows), len(indicator_rows), "✅", path))

# ---- Final report: did we get everything we were promised? -------------
print("\n================ BRONZE LANDING REPORT ================")
for name, got, expected, status, path in summary:
    print(f"{status} {name:11} received {got:>6} of {expected:>6} -> {path}")